# LeetCode 793 — Preimage Size of Factorial Zeroes Function bằng Python

- Link: https://leetcode.com/problems/preimage-size-of-factorial-zeroes-function/
- Độ khó: **Hard**
- Chủ đề: Toán (công thức Legendre), Tìm kiếm nhị phân trên đáp án, Hàm đơn điệu, Hệ cơ số đặc biệt

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def preimageSizeFZF(self, k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Gọi `f(x)` là số chữ số `0` ở cuối của `x!`. Ví dụ `f(5) = 1` vì `5! = 120`. Cho `k`, hỏi có **bao nhiêu** số nguyên không âm `x` sao cho `f(x) = k`?

Điều thú vị: đáp án **chỉ có thể là 0 hoặc 5**! Bài này dạy ta cách phân tích một hàm đơn điệu "nhảy bậc", và cách dùng **tìm kiếm nhị phân trên một miền rất lớn** (`x` có thể tới cỡ `5·10⁹`).

Sau notebook này bạn sẽ:

1. Tính `f(x)` bằng **công thức Legendre**: `f(x) = ⌊x/5⌋ + ⌊x/25⌋ + ⌊x/125⌋ + ...`.
2. Hiểu vì sao `f` **không giảm**, hằng trên từng khối 5 số liên tiếp, và có những giá trị `k` bị **nhảy qua**.
3. Đếm số nghiệm bằng **hai lần tìm kiếm nhị phân** (cận trái của `k` và `k+1`).
4. Rút gọn thành **một lần nhị phân** (đáp án ∈ {0, 5}).
5. Giải bằng **phân tích theo cơ số (5ⁱ−1)/4** mà không cần nhị phân.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Định nghĩa `f(x)` = số chữ số 0 liên tiếp ở cuối `x!` (quy ước `0! = 1`). Cho số nguyên `k`, trả về số lượng số nguyên không âm `x` thoả `f(x) = k`.

**Ví dụ 1**

```text
Input: k = 0
Output: 5
Giải thích: 0!, 1!, 2!, 3!, 4! đều không có số 0 ở cuối.
```

**Ví dụ 2**

```text
Input: k = 5
Output: 0
Giải thích: không có x nào mà x! có đúng 5 chữ số 0 ở cuối.
```

**Ví dụ 3**

```text
Input: k = 3
Output: 5
```

**Ràng buộc**

- `0 <= k <= 10⁹`

In [1]:
EXAMPLES = [(0, 5), (5, 0), (3, 5), (1000000000, 5), (6, 5), (11, 0)]

from math import factorial
for x in range(0, 31, 5):
    s = str(factorial(x))
    print(f"{x:2}! = {s[-12:]:>12} ... số 0 cuối = {len(s) - len(s.rstrip('0'))}")

 0! =            1 ... số 0 cuối = 0
 5! =          120 ... số 0 cuối = 1
10! =      3628800 ... số 0 cuối = 2
15! = 307674368000 ... số 0 cuối = 3
20! = 008176640000 ... số 0 cuối = 4
25! = 985984000000 ... số 0 cuối = 6
30! = 308480000000 ... số 0 cuối = 7


## 3. Kiến thức nền

### 3.1. Số 0 ở cuối = số cặp (2, 5)

Mỗi chữ số 0 ở cuối là một thừa số `10 = 2 × 5`. Trong `x!`, số thừa số 2 luôn nhiều hơn số thừa số 5, nên **số 0 ở cuối = số mũ của 5 trong `x!`**.

### 3.2. Công thức Legendre

Số mũ của số nguyên tố `p` trong `x!` là `⌊x/p⌋ + ⌊x/p²⌋ + ⌊x/p³⌋ + ...`. Với `p = 5`: mỗi bội của 5 góp 1, mỗi bội của 25 góp thêm 1, mỗi bội của 125 góp thêm 1, ...

In [2]:
def f(x):
    """Số chữ số 0 ở cuối của x!"""
    cnt = 0
    while x:
        x //= 5
        cnt += x
    return cnt

def f_slow(x):
    s = str(factorial(x))
    return len(s) - len(s.rstrip('0'))

for x in range(0, 200):
    assert f(x) == f_slow(x)
print([f(x) for x in range(0, 31)])
print("f(10^9) =", f(10**9))

[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 7]
f(10^9) = 249999998


Vòng lặp `x //= 5; cnt += x` tính lần lượt `⌊x/5⌋, ⌊x/25⌋, ...` vì `⌊⌊x/5⌋/5⌋ = ⌊x/25⌋`. Chạy `O(log₅ x)` bước.

### 3.3. Tìm kiếm nhị phân "cận trái" trên hàm không giảm

Với hàm không giảm `g`, tìm `x` nhỏ nhất mà `g(x) >= k`: `lo, hi`; nếu `g(mid) >= k` thì `hi = mid`, ngược lại `lo = mid + 1`. Đây là `bisect_left` trên một "mảng ảo".

In [3]:
def lower_bound(g, k, lo, hi):
    """x nhỏ nhất trong [lo, hi] với g(x) >= k (giả sử g(hi) >= k)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if g(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(lower_bound(f, 6, 0, 100), "->", f(lower_bound(f, 6, 0, 100)))

25 -> 6


### 3.4. Biểu diễn số trong một hệ cơ số "lạ"

Ta quen viết `n` trong cơ số 10 bằng cách chia lấy thương lần lượt cho `10^i` từ lớn tới nhỏ. Tổng quát: với dãy "trọng số" tăng nhanh `w₀ < w₁ < ...`, ta có thể phân tích tham lam `n = Σ dᵢ·wᵢ`. Cách 4 dùng trọng số `wᵢ = (5ⁱ − 1)/4 = 1, 6, 31, 156, ...`.

In [4]:
W = [(5**i - 1) // 4 for i in range(1, 15)]
print(W[:8])
# quan hệ truy hồi: w_{i+1} = 5*w_i + 1
print(all(W[i + 1] == 5 * W[i] + 1 for i in range(len(W) - 1)))

[1, 6, 31, 156, 781, 3906, 19531, 97656]
True


## 4. Ý tưởng nền tảng / trực giác

### 4.1. `f` hằng trên từng khối 5 số

`f(x)` chỉ tăng khi `x` là bội của 5. Vì vậy `f(5m) = f(5m+1) = ... = f(5m+4)`. Mỗi giá trị đạt được của `f` được đạt bởi **ít nhất** 5 số liên tiếp.

### 4.2. `f` nhảy nhiều bậc tại bội của 25

Khi `x` là bội của 25, `f` tăng ít nhất 2; bội của 125 → tăng ít nhất 3. Các giá trị bị nhảy qua sẽ có **0** nghiệm.

```text
x    : 0..4  5..9  10..14  15..19  20..24  25..29  30..34
f(x) :  0     1      2       3       4       6       7
```

`f` nhảy từ 4 lên 6 → `k = 5` không có nghiệm.

### 4.3. Vì sao không thể là 10 nghiệm?

Nếu `f(x) = k` có nghiệm, các nghiệm tạo thành **một đoạn liên tiếp** (vì `f` không giảm). Đoạn đó bắt đầu ở một bội của 5 (`5m`) và kết thúc ở `5m + 4` vì tại `5m + 5` hàm tăng ít nhất 1. → đúng 5 nghiệm.

### 4.4. Tính tay

- `k = 3`: `f(15..19) = 3` → 5 nghiệm.
- `k = 5`: `f(24) = 4`, `f(25) = 6` → 0 nghiệm.
- `k = 11`: `f(45..49) = 10`, `f(50) = 12` → 0 nghiệm.

In [5]:
from collections import Counter
cnt = Counter(f(x) for x in range(0, 500))
print("các k bị nhảy qua (k < f(499)):", [k for k in range(f(499)) if cnt[k] == 0])
print("số nghiệm xuất hiện:", sorted(set(cnt[k] for k in range(f(499)))))

các k bị nhảy qua (k < f(499)): [5, 11, 17, 23, 29, 30, 36, 42, 48, 54, 60, 61, 67, 73, 79, 85, 91, 92, 98, 104, 110, 116]
số nghiệm xuất hiện: [0, 5]


## 5. Cách 1 — Brute Force: duyệt `x` tăng dần

**Ý tưởng.** Duyệt `x = 0, 1, 2, ...`, đếm những `x` có `f(x) = k`, dừng khi `f(x) > k`. Tính `f` tăng dần bằng cách cộng số mũ của 5 trong `x` (để mỗi bước `O(1)` trung bình).

In [6]:
def v5(x):
    """số mũ của 5 trong x (x > 0)"""
    c = 0
    while x % 5 == 0:
        x //= 5
        c += 1
    return c

def preimage_brute(k):
    count = 0
    x, fx = 0, 0                     # f(0) = 0
    while fx <= k:
        if fx == k:
            count += 1
        x += 1
        fx += v5(x)                  # f(x) = f(x-1) + v5(x)
    return count

for k, exp in EXAMPLES[:3] + EXAMPLES[4:]:
    print(k, preimage_brute(k), "(kỳ vọng", exp, ")")
    assert preimage_brute(k) == exp

0 5 (kỳ vọng 5 )
5 0 (kỳ vọng 0 )
3 5 (kỳ vọng 5 )
6 5 (kỳ vọng 5 )
11 0 (kỳ vọng 0 )


**Độ phức tạp.** `x` cần chạy tới khoảng `4k..5k` (vì `f(x) ≈ x/4`) → **`O(k)`**. Với `k = 10⁹` là ~4 tỷ vòng lặp → quá chậm.

**Nút thắt.** Ta không cần duyệt từng `x`: `f` không giảm, nên vị trí đầu tiên `f(x) ≥ k` tìm được bằng **nhị phân**.

## 6. Bước đột phá

**Insight 1 — Đếm bằng hai cận.** Vì `f` không giảm:

```text
#{x : f(x) = k} = L(k+1) − L(k),   trong đó L(k) = min{x : f(x) ≥ k}
```

**Insight 2 — Cận trên cho nhị phân.** `f(x) ≥ x/5`, nên `f(5k) ≥ k` → `L(k) ≤ 5k`. Tìm trong `[0, 5k]` (hoặc `5k + 5` cho chắc với `k+1`).

**Insight 3 — Đáp án chỉ 0 hoặc 5.** (Mục 4.3) → chỉ cần kiểm tra **có tồn tại** `x` với `f(x) = k` hay không: `f(L(k)) == k ? 5 : 0`.

**Insight 4 — Bỏ hẳn nhị phân.** Ta có `f(5ⁱ) = (5ⁱ − 1)/4 = wᵢ`. Các giá trị `k` bị nhảy qua có cấu trúc đệ quy theo `wᵢ`: phân tích tham lam `k` theo các `wᵢ`, nếu có "chữ số" bằng 5 thì `k` bị nhảy qua.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai lần tìm kiếm nhị phân

In [7]:
def L(k):
    """x nhỏ nhất với f(x) >= k."""
    lo, hi = 0, 5 * k + 5
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

def preimage_two_bs(k):
    return L(k + 1) - L(k)

for k, exp in EXAMPLES:
    assert preimage_two_bs(k) == exp, k
print("Hai lần nhị phân: OK")
print("L(5) =", L(5), " L(6) =", L(6), " -> số nghiệm =", L(6) - L(5))

Hai lần nhị phân: OK
L(5) = 25  L(6) = 25  -> số nghiệm = 0


Code nộp LeetCode cho cách 2:

In [8]:
class SolutionTwoBS:
    def preimageSizeFZF(self, k: int) -> int:
        def zeros(x):
            c = 0
            while x:
                x //= 5
                c += x
            return c

        def left(t):
            lo, hi = 0, 5 * t + 5
            while lo < hi:
                mid = (lo + hi) // 2
                if zeros(mid) >= t:
                    hi = mid
                else:
                    lo = mid + 1
            return lo

        return left(k + 1) - left(k)

for k, exp in EXAMPLES:
    assert SolutionTwoBS().preimageSizeFZF(k) == exp
print("SolutionTwoBS: OK")

SolutionTwoBS: OK


**Độ phức tạp cách 2:** mỗi lần nhị phân `O(log k)` bước, mỗi bước tính `f` mất `O(log k)` → **`O(log² k)`**. Bộ nhớ `O(1)`.

### 7.2. Cách 3 — Một lần nhị phân: có tồn tại `x` với `f(x) = k`?

Nhị phân trực tiếp: nếu `f(mid) == k` → trả `5`; nếu `f(mid) < k` → `lo = mid + 1`; nếu `> k` → `hi = mid - 1`. Hết đoạn mà không gặp → `0`.

In [9]:
def preimage_one_bs(k):
    lo, hi = 0, 5 * k
    while lo <= hi:
        mid = (lo + hi) // 2
        z = f(mid)
        if z == k:
            return 5
        if z < k:
            lo = mid + 1
        else:
            hi = mid - 1
    return 0

for k, exp in EXAMPLES:
    assert preimage_one_bs(k) == exp, k
print("Một lần nhị phân: OK")

Một lần nhị phân: OK


Có thể thu hẹp không gian: chỉ nhị phân trên các bội của 5 (`x = 5m`), vì `f(5m) = m + f(m)`. Khi đó tìm `m` với `m + f(m) = k`.

In [10]:
class SolutionOneBS:
    def preimageSizeFZF(self, k: int) -> int:
        def zeros(x):
            c = 0
            while x:
                x //= 5
                c += x
            return c

        lo, hi = 0, k                      # tìm m với f(5m) = m + zeros(m) = k
        while lo <= hi:
            m = (lo + hi) // 2
            z = m + zeros(m)
            if z == k:
                return 5
            if z < k:
                lo = m + 1
            else:
                hi = m - 1
        return 0

for k, exp in EXAMPLES:
    assert SolutionOneBS().preimageSizeFZF(k) == exp
print("SolutionOneBS: OK")

SolutionOneBS: OK


Vì sao `hi = k` đủ? `f(5m) = m + f(m) ≥ m`, nên nếu `m > k` thì `f(5m) > k`. Và `f(5m) = 5m/5 + f(m) = m + f(m)` vì `⌊5m/5ⁱ⌋ = ⌊m/5^{i-1}⌋`.

**Độ phức tạp cách 3:** `O(log² k)` thời gian, `O(1)` bộ nhớ.

### 7.3. Cách 4 — Phân tích theo trọng số `wᵢ = (5ⁱ − 1)/4`

**Quan sát.** `f(5ⁱ) = 5^{i-1} + 5^{i-2} + ... + 1 = (5ⁱ − 1)/4 = wᵢ`.

Hãy xét các giá trị `f` trên đoạn `[0, 5ⁱ⁺¹)`: đoạn này chia thành 5 khối, mỗi khối dài `5ⁱ`. Trong khối thứ `d` (`d = 0..4`), `f(d·5ⁱ + r) = d·wᵢ + f(r)` (với `0 ≤ r < 5ⁱ`; chứng minh: các bội của `5^j` với `j ≤ i` trong `d·5ⁱ + r` gồm các bội trong `d·5ⁱ` cộng các bội trong `r`). Vậy tập giá trị đạt được trên `[0, 5ⁱ⁺¹)` là

```text
V_{i+1} = { d·wᵢ + v : d = 0..4, v ∈ V_i }
```

và mọi giá trị trong `V_i` đều `< wᵢ` (vì `f(5ⁱ − 1) = wᵢ − i < wᵢ`). Quan trọng hơn: giá trị `5·wᵢ` (đáng lẽ là "khối thứ 5") **không** được đạt bởi `f`, vì `f(5ⁱ⁺¹) = wᵢ₊₁ = 5wᵢ + 1` — nhảy qua `5wᵢ`.

**Thuật toán.** Phân tích tham lam `k` từ trọng số lớn nhất xuống: `d = k // wᵢ`; nếu `d == 5` → `k` không đạt được → `0`; ngược lại `k %= wᵢ`. Nếu đi hết mà không gặp `d == 5` → `5`.

In [11]:
def preimage_weights(k):
    w = 1
    while w <= k:                # trọng số lớn nhất <= k
        w = 5 * w + 1
    while w > 0:
        d = k // w
        if d == 5:
            return 0
        k %= w
        w = (w - 1) // 5         # trọng số nhỏ hơn
    return 5

for k, exp in EXAMPLES:
    assert preimage_weights(k) == exp, k
print("Phân tích theo trọng số: OK")

Phân tích theo trọng số: OK


Thử với `k = 5`: trọng số lớn nhất `≤ 5` là `1`... vòng `while w <= k` cho `w = 31`? — Hãy theo dõi: `w = 1 ≤ 5 → 6`, `6 > 5` dừng; `w = 6`: `d = 5 // 6 = 0`, `k = 5`; `w = 1`: `d = 5` → **0**. ✔

Thử `k = 11`: `w`: 1 → 6 → 31; `w = 31`: `d = 0`; `w = 6`: `d = 1`, `k = 5`; `w = 1`: `d = 5` → **0**. ✔

In [12]:
class SolutionWeights:
    def preimageSizeFZF(self, k: int) -> int:
        w = 1
        while w <= k:
            w = 5 * w + 1
        while w:
            if k // w == 5:
                return 0
            k %= w
            w = (w - 1) // 5
        return 5

for k, exp in EXAMPLES:
    assert SolutionWeights().preimageSizeFZF(k) == exp
print("SolutionWeights: OK")

SolutionWeights: OK


**Độ phức tạp cách 4:** `O(log₅ k)` thời gian, `O(1)` bộ nhớ — nhanh nhất.

Lưu ý: "chữ số" `d` trong phân tích tham lam luôn `≤ 5` vì `wᵢ₊₁ = 5wᵢ + 1 > 5wᵢ`, nên phần dư sau khi chia cho `wᵢ₊₁` luôn `< 5wᵢ + 1`, tức `≤ 5wᵢ`. `d = 5` xảy ra đúng khi phần dư bằng `5wᵢ`... thực ra khi phần dư thuộc `[5wᵢ, 5wᵢ]` = đúng một giá trị, rồi `k %= wᵢ` = 0. Vậy một cách tương đương: `k` bị nhảy qua ⇔ tại một bước nào đó phần dư **bằng đúng** `5wᵢ`.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (hai lần nhị phân) với `k = 5`

In [13]:
def trace_L(k):
    lo, hi = 0, 5 * k + 5
    step = 0
    print(f"Tìm L({k}) = min x với f(x) >= {k}")
    while lo < hi:
        mid = (lo + hi) // 2
        z = f(mid)
        print(f"  bước {step}: lo={lo:3} hi={hi:3} mid={mid:3} f(mid)={z:2} -> {'hi=mid' if z >= k else 'lo=mid+1'}")
        if z >= k:
            hi = mid
        else:
            lo = mid + 1
        step += 1
    print(f"  => L({k}) = {lo}, f(L) = {f(lo)}")
    return lo

a = trace_L(5)
b = trace_L(6)
print("số nghiệm của k=5:", b - a)

Tìm L(5) = min x với f(x) >= 5
  bước 0: lo=  0 hi= 30 mid= 15 f(mid)= 3 -> lo=mid+1
  bước 1: lo= 16 hi= 30 mid= 23 f(mid)= 4 -> lo=mid+1
  bước 2: lo= 24 hi= 30 mid= 27 f(mid)= 6 -> hi=mid
  bước 3: lo= 24 hi= 27 mid= 25 f(mid)= 6 -> hi=mid
  bước 4: lo= 24 hi= 25 mid= 24 f(mid)= 4 -> lo=mid+1
  => L(5) = 25, f(L) = 6
Tìm L(6) = min x với f(x) >= 6
  bước 0: lo=  0 hi= 35 mid= 17 f(mid)= 3 -> lo=mid+1
  bước 1: lo= 18 hi= 35 mid= 26 f(mid)= 6 -> hi=mid
  bước 2: lo= 18 hi= 26 mid= 22 f(mid)= 4 -> lo=mid+1
  bước 3: lo= 23 hi= 26 mid= 24 f(mid)= 4 -> lo=mid+1
  bước 4: lo= 25 hi= 26 mid= 25 f(mid)= 6 -> hi=mid
  => L(6) = 25, f(L) = 6
số nghiệm của k=5: 0


Bảng tóm tắt: `L(5) = 25` (vì `f(24) = 4 < 5`, `f(25) = 6 ≥ 5`), `L(6) = 25` → `25 − 25 = 0` nghiệm. Với `k = 6`: `L(6) = 25`, `L(7) = 30` → 5 nghiệm (`25..29`).

### 8.2. Trace cách 4 với vài giá trị `k`

In [14]:
def trace_weights(k):
    orig = k
    w = 1
    while w <= k:
        w = 5 * w + 1
    digits = []
    while w:
        d = k // w
        digits.append((w, d))
        if d == 5:
            print(f"k={orig}: các (trọng số, chữ số) = {digits} -> gặp chữ số 5 => 0")
            return 0
        k %= w
        w = (w - 1) // 5
    print(f"k={orig}: các (trọng số, chữ số) = {digits} -> 5")
    return 5

for k in [5, 6, 11, 30, 31, 155, 156, 1000]:
    r = trace_weights(k)
    assert r == preimage_brute(k)

k=5: các (trọng số, chữ số) = [(6, 0), (1, 5)] -> gặp chữ số 5 => 0
k=6: các (trọng số, chữ số) = [(31, 0), (6, 1), (1, 0)] -> 5
k=11: các (trọng số, chữ số) = [(31, 0), (6, 1), (1, 5)] -> gặp chữ số 5 => 0
k=30: các (trọng số, chữ số) = [(31, 0), (6, 5)] -> gặp chữ số 5 => 0
k=31: các (trọng số, chữ số) = [(156, 0), (31, 1), (6, 0), (1, 0)] -> 5
k=155: các (trọng số, chữ số) = [(156, 0), (31, 5)] -> gặp chữ số 5 => 0
k=156: các (trọng số, chữ số) = [(781, 0), (156, 1), (31, 0), (6, 0), (1, 0)] -> 5
k=1000: các (trọng số, chữ số) = [(3906, 0), (781, 1), (156, 1), (31, 2), (6, 0), (1, 1)] -> 5


Bảng cho `k = 30`:

| trọng số `w` | `k` trước | chữ số `d = k // w` | `k` sau |
|---|---|---|---|
| 31 | 30 | 0 | 30 |
| 6 | 30 | 5 | — → **0 nghiệm** |

Thật vậy `f(124) = 24 + 4 = 28`, `f(125) = 31` → `29, 30` bị nhảy qua.

In [15]:
print("f(120..126):", [f(x) for x in range(120, 127)])

f(120..126): [28, 28, 28, 28, 28, 31, 31]


## 9. Code nộp LeetCode

Chọn cách 3 (một lần nhị phân trên `m`) — dễ hiểu, dễ chứng minh, đủ nhanh:

In [16]:
class Solution:
    def preimageSizeFZF(self, k: int) -> int:
        def zeros(x: int) -> int:          # số 0 ở cuối của x!
            c = 0
            while x:
                x //= 5
                c += x
            return c

        lo, hi = 0, 5 * k                  # f(5k) >= k nên nghiệm (nếu có) nằm trong [0, 5k]
        while lo <= hi:
            mid = (lo + hi) // 2
            z = zeros(mid)
            if z == k:
                return 5                   # có nghiệm -> đúng 5 nghiệm
            if z < k:
                lo = mid + 1
            else:
                hi = mid - 1
        return 0

for k, exp in EXAMPLES:
    assert Solution().preimageSizeFZF(k) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đơn điệu).** `f(x+1) − f(x) = v₅(x+1) ≥ 0` → `f` không giảm.

**Bổ đề 2 (khối 5).** `v₅(x+1) = 0` khi `x + 1` không chia hết cho 5 → `f(5m) = f(5m+1) = ... = f(5m+4)`; và `f(5m+5) ≥ f(5m+4) + 1`.

**Định lý.** Với mọi `k`, `#{x : f(x) = k} ∈ {0, 5}`.
*Chứng minh.* Tập nghiệm là một đoạn liên tiếp (Bổ đề 1). Nếu khác rỗng, gọi `x₀` là nghiệm nhỏ nhất. `x₀` phải là bội của 5 (nếu không, `f(x₀ − 1) = f(x₀)` mâu thuẫn tính nhỏ nhất — trừ `x₀ = 0`, cũng là bội của 5). Theo Bổ đề 2, `x₀..x₀+4` là nghiệm và `x₀ + 5` không phải. ∎

**Cách 2:** `L(k+1) − L(k)` = số `x` với `k ≤ f(x) < k+1` = số nghiệm. Cận `hi = 5k + 5` hợp lệ vì `f(5(k+1)) ≥ k + 1`.

**Cách 3:** nhị phân tìm một nghiệm trong `[0, 5k]` trên hàm không giảm — nếu tồn tại thì chắc chắn được tìm thấy (tính chất của nhị phân trên dãy đã sắp xếp).

**Cách 4:** Quy nạp: tập giá trị đạt được trên `[0, 5ⁱ⁺¹)` là `{d·wᵢ + v : 0 ≤ d ≤ 4, v ∈ Vᵢ}`, trong đó mọi `v ∈ Vᵢ` thoả `v < wᵢ` (vì `f(5ⁱ − 1) < f(5ⁱ) = wᵢ`). Do đó biểu diễn `k = d·wᵢ + v` với `0 ≤ v < wᵢ` là duy nhất (`d = k // wᵢ`, `v = k % wᵢ`), và `k` đạt được ⇔ `d ≤ 4` và `v` đạt được. Với `k < wᵢ₊₁ = 5wᵢ + 1` thì `d ≤ 5`, và `d = 5` ⇔ `k = 5wᵢ` — giá trị bị nhảy qua. Thuật toán chính là áp dụng đệ quy điều này.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt `x` | `O(k)` | `O(1)` |
| 2. Hai lần nhị phân | `O(log² k)` | `O(1)` |
| 3. Một lần nhị phân | `O(log² k)` | `O(1)` |
| 4. Trọng số `wᵢ` | `O(log k)` | `O(1)` |

Với `k = 10⁹`: `log₂(5·10⁹) ≈ 33` bước nhị phân × `log₅(5·10⁹) ≈ 14` phép chia ≈ 460 phép tính.

In [17]:
import time
for name, fn in [("two_bs", preimage_two_bs), ("one_bs", preimage_one_bs), ("weights", preimage_weights)]:
    t0 = time.perf_counter()
    for k in range(10**9 - 2000, 10**9):
        fn(k)
    print(f"{name:8}: 2000 truy vấn k ~ 1e9 mất {(time.perf_counter() - t0) * 1000:.1f} ms")

two_bs  : 2000 truy vấn k ~ 1e9 mất 161.8 ms


one_bs  : 2000 truy vấn k ~ 1e9 mất 48.5 ms
weights : 2000 truy vấn k ~ 1e9 mất 4.1 ms


## 12. Bộ test

In [18]:
import random

methods = {
    "brute": preimage_brute,
    "two_bs": preimage_two_bs,
    "one_bs": preimage_one_bs,
    "weights": preimage_weights,
    "SolutionTwoBS": lambda k: SolutionTwoBS().preimageSizeFZF(k),
    "SolutionOneBS": lambda k: SolutionOneBS().preimageSizeFZF(k),
    "SolutionWeights": lambda k: SolutionWeights().preimageSizeFZF(k),
    "Solution": lambda k: Solution().preimageSizeFZF(k),
}
fast = [m for m in methods if m != "brute"]

# 12.1 Ví dụ đề + edge cases
for k, exp in EXAMPLES:
    for name in fast:
        assert methods[name](k) == exp, (name, k)
for k in [0, 1, 4, 5, 6, 10, 11, 12, 28, 29, 30, 31]:
    ref = preimage_brute(k)
    for name, fn in methods.items():
        assert fn(k) == ref, (name, k)
print("Ví dụ + edge cases: OK")

# 12.2 Toàn bộ k từ 0..3000: đối chiếu với bảng đếm trực tiếp
N = 15000
table = Counter(f(x) for x in range(N))
for k in range(0, 3000):
    ref = table[k]
    for name in fast:
        assert methods[name](k) == ref, (name, k)
print("k = 0..2999: khớp bảng đếm")

# 12.3 Ngẫu nhiên có seed, k lớn: các cách nhanh khớp nhau
rnd = random.Random(793)
zero_cnt = 0
for _ in range(3000):
    k = rnd.randint(0, 10**9)
    vals = {name: methods[name](k) for name in fast}
    assert len(set(vals.values())) == 1, (k, vals)
    zero_cnt += vals["Solution"] == 0
print("3000 k ngẫu nhiên lớn: khớp; tỉ lệ k bị nhảy qua ≈", zero_cnt / 3000)

# 12.4 Giá trị k = f(x) luôn có 5 nghiệm; k = f(25m)-1 (nếu nhảy) có 0
for _ in range(500):
    x = rnd.randint(0, 4 * 10**9)
    assert Solution().preimageSizeFZF(f(x)) == 5
    m = rnd.randint(1, 10**8)
    if f(25 * m) - f(25 * m - 1) >= 2:
        assert Solution().preimageSizeFZF(f(25 * m) - 1) == 0
print("k = f(x): luôn 5; giá trị bị nhảy: 0 — OK")

Ví dụ + edge cases: OK
k = 0..2999: khớp bảng đếm


3000 k ngẫu nhiên lớn: khớp; tỉ lệ k bị nhảy qua ≈ 0.201
k = f(x): luôn 5; giá trị bị nhảy: 0 — OK


Tỉ lệ `k` bị nhảy qua ≈ 20%: trong mỗi 5 bước của `f` trên khối 25, có một bước tăng ≥ 2.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cận trên nhị phân quá nhỏ (ví dụ `hi = k`)

**Code sai:**

In [19]:
def wrong_hi(k):
    lo, hi = 0, k                    # SAI: f(x) ≈ x/4 nên x cần cỡ 4k
    while lo <= hi:
        mid = (lo + hi) // 2
        z = f(mid)
        if z == k:
            return 5
        if z < k:
            lo = mid + 1
        else:
            hi = mid - 1
    return 0

for k in [1, 3, 6, 100]:
    print(k, "sai:", wrong_hi(k), " đúng:", Solution().preimageSizeFZF(k))

1 sai: 0  đúng: 5
3 sai: 0  đúng: 5
6 sai: 0  đúng: 5
100 sai: 0  đúng: 5


**Vì sao sai:** nghiệm `x` của `f(x) = k` nằm quanh `4k` (vì `f(x) ≈ x/5 + x/25 + ... = x/4`). Tìm trong `[0, k]` không bao giờ thấy.

**Code đúng:** `hi = 5k` (vì `f(5k) ≥ k`). (Riêng cách nhị phân trên `m` với `x = 5m` thì `hi = k` là đúng!)

### Lỗi 2 — Tính `x!` thật rồi đếm số 0

**Code sai:**

In [20]:
def wrong_factorial(k, limit=2000):
    cnt = 0
    for x in range(limit):
        s = str(factorial(x))        # SAI: x! khổng lồ, rất chậm
        if len(s) - len(s.rstrip('0')) == k:
            cnt += 1
    return cnt

t0 = time.perf_counter()
print("k=3:", wrong_factorial(3, limit=600), f"({(time.perf_counter() - t0)*1000:.0f} ms cho x < 600)")

k=3: 5 (8 ms cho x < 600)


**Vì sao sai:** `x!` có `O(x log x)` chữ số; với `x` cỡ `4·10⁹` thì không thể lưu trữ, chưa nói tính toán. Hơn nữa `limit` cứng có thể bỏ sót.

**Code đúng:** công thức Legendre `f(x) = Σ ⌊x/5ⁱ⌋`.

### Lỗi 3 — Đếm số 0 bằng `x // 5` (quên bội của 25, 125, ...)

**Code sai:**

In [21]:
def wrong_f(x):
    return x // 5                    # SAI: thiếu x//25 + x//125 + ...

def wrong_preimage(k):
    lo, hi = 0, 5 * k
    while lo <= hi:
        mid = (lo + hi) // 2
        z = wrong_f(mid)
        if z == k:
            return 5
        if z < k:
            lo = mid + 1
        else:
            hi = mid - 1
    return 0

for k in [5, 6, 11, 30]:
    print(k, "sai:", wrong_preimage(k), " đúng:", Solution().preimageSizeFZF(k))

5 sai: 5  đúng: 0
6 sai: 5  đúng: 5
11 sai: 5  đúng: 0
30 sai: 5  đúng: 0


**Vì sao sai:** `25 = 5 × 5` đóng góp **hai** thừa số 5. Với `wrong_f`, mọi `k` đều có nghiệm → luôn trả 5.

**Code đúng:** vòng `while x: x //= 5; c += x`.

### Lỗi 4 — Trả về `L(k+1) − L(k)` với `L` tìm "phần tử bằng k" thay vì "≥ k"

Nếu `L(k)` được viết là "x nhỏ nhất với `f(x) == k`" thì với `k` bị nhảy qua hàm không xác định (có thể trả `-1` hoặc giá trị rác). Luôn dùng điều kiện `>=` cho cận trái.

## 14. Mẹo & nhận dạng pattern

- **Số 0 ở cuối của `n!`** → công thức Legendre với `p = 5` (LeetCode 172 *Factorial Trailing Zeroes*).
- **"Đếm số x với g(x) = k" khi g không giảm** → `L(k+1) − L(k)` bằng hai lần `bisect_left`.
- **Nhị phân trên miền giá trị khổng lồ:** cần một cận trên chứng minh được (`f(5k) ≥ k`).
- **Đáp án chỉ có vài giá trị** (ở đây 0/5) → bài toán quyết định "có tồn tại không", đơn giản hơn bài đếm.
- **Cấu trúc tự đồng dạng cơ số 5** → nghĩ tới phân tích theo `wᵢ = (5ⁱ−1)/4`.

## 15. Biến thể

1. **172. Factorial Trailing Zeroes:** chỉ cần tính `f(n)`.
2. **Số 0 ở cuối trong cơ số `b`:** phân tích `b = Π pᵢ^{eᵢ}`; số 0 = `min_i ⌊Legendre(n, pᵢ)/eᵢ⌋`.
3. **x nhỏ nhất có ít nhất `k` số 0 ở cuối:** chính là `L(k)` (bài "smallest factorial number" trên GFG).
4. **Đếm các `k ≤ K` bị nhảy qua:** `K + 1 − (số giá trị f đạt được trong [0, K])`.
5. **Số 0 ở cuối của `C(n, r)` hoặc tích một dãy:** cộng/trừ Legendre.

Minh hoạ biến thể 2 — số 0 ở cuối của `n!` trong cơ số 12 (= 2²·3):

In [22]:
def legendre(n, p):
    c = 0
    while n:
        n //= p
        c += n
    return c

def trailing_zeros_base(n, base):
    # phân tích base
    fac = {}
    b, d = base, 2
    while d * d <= b:
        while b % d == 0:
            fac[d] = fac.get(d, 0) + 1
            b //= d
        d += 1
    if b > 1:
        fac[b] = fac.get(b, 0) + 1
    return min(legendre(n, p) // e for p, e in fac.items())

def trailing_zeros_base_slow(n, base):
    v, c = factorial(n), 0
    while v % base == 0:
        v //= base; c += 1
    return c

for n in range(1, 60):
    for base in (10, 12, 16, 9):
        assert trailing_zeros_base(n, base) == trailing_zeros_base_slow(n, base)
print("20! trong cơ số 12 có", trailing_zeros_base(20, 12), "số 0 ở cuối")

20! trong cơ số 12 có 8 số 0 ở cuối


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt `x` | đếm trực tiếp | `O(k)` | `O(1)` | `k` nhỏ, đối chiếu |
| 2. Hai lần nhị phân | `L(k+1) − L(k)` | `O(log² k)` | `O(1)` | tổng quát cho mọi hàm không giảm |
| 3. Một lần nhị phân | có nghiệm ⇒ 5 | `O(log² k)` | `O(1)` | **nộp bài**, ngắn |
| 4. Trọng số `(5ⁱ−1)/4` | cấu trúc cơ số 5 | `O(log k)` | `O(1)` | nhanh nhất, đẹp về toán |

## 17. Bài tập

**Bài 1 (đoán output).** `preimageSizeFZF(24)`, `preimageSizeFZF(29)`, `preimageSizeFZF(31)` = ?

**Bài 2 (tính tay).** Tính `f(100)`, `f(124)`, `f(125)`. Những `k` nào trong khoảng `[f(124), f(125)]` bị nhảy qua?

**Bài 3 (điền TODO).** Hoàn thiện `smallest_x(k)` trả về `x` nhỏ nhất sao cho `x!` có **ít nhất** `k` số 0 ở cuối.

**Bài 4 (tự viết lại).** Viết `count_skipped(K)` = số giá trị `k ∈ [0, K]` có đáp án 0, trong `O(log² K)` hoặc nhanh hơn (gợi ý: số giá trị đạt được = số khối 5 có `f ≤ K`).

**Bài 5 (mini challenge).** Tổng quát hoá: cho số nguyên tố `p` và `k`, đếm số `x` với `v_p(x!) = k`. Viết `preimage_p(k, p)` (đáp án ∈ {0, p}).

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def smallest_x(k):
    # TODO: nhị phân cận trái trên [0, 5k] với điều kiện f(x) >= k
    pass

# Bài 4 — TODO
def count_skipped(K):
    # TODO: m* = số m >= 0 với f(5m) <= K (nhị phân); đáp án = (K + 1) - m*
    pass

# Bài 5 — TODO
def preimage_p(k, p):
    # TODO: thay 5 bằng p trong Legendre và trong cận trên (p*k)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê `f` quanh `x = 100..130`.
- Mức 2: `f(100) = 24` → `k = 24` có nghiệm.
- Mức 3: `f(124) = 28`, `f(125) = 31` → `29`, `30` bị nhảy; `31` có nghiệm.

**Bài 2.**
- Mức 1: `f(100) = 20 + 4 = 24`.
- Mức 2: `f(124) = 24 + 4 = 28`; `f(125) = 25 + 5 + 1 = 31`.
- Mức 3: bị nhảy: `29, 30`.

**Bài 3.**
- Mức 1: chính là `L(k)`.
- Mức 2: `hi = 5k` đủ.
- Mức 3: `k = 0` → `0`.

**Bài 4.**
- Mức 1: mỗi giá trị đạt được ứng với đúng một khối `[5m, 5m+4]`.
- Mức 2: số giá trị đạt được trong `[0, K]` = số `m ≥ 0` với `f(5m) ≤ K`.
- Mức 3: `f(5m) = m + f(m)` tăng ngặt theo `m` → nhị phân.

**Bài 5.**
- Mức 1: `v_p(x!) = Σ ⌊x/pⁱ⌋`.
- Mức 2: hằng trên khối `p` số liên tiếp → đáp án ∈ {0, p}.
- Mức 3: `hi = p·k`.

## 19. Đáp án tham khảo

In [24]:
S = Solution()
# Bài 1
assert S.preimageSizeFZF(24) == 5
assert S.preimageSizeFZF(29) == 0
assert S.preimageSizeFZF(31) == 5

# Bài 2
assert (f(100), f(124), f(125)) == (24, 28, 31)
assert [k for k in range(28, 32) if S.preimageSizeFZF(k) == 0] == [29, 30]

# Bài 3
def smallest_x_ans(k):
    lo, hi = 0, 5 * k
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for k in range(0, 500):
    x = smallest_x_ans(k)
    assert f(x) >= k and (x == 0 or f(x - 1) < k)

# Bài 4
def count_skipped_ans(K):
    lo, hi = 0, K + 1            # số m với m + f(m) <= K; m + f(m) >= m nên m <= K
    while lo < hi:               # tìm m nhỏ nhất với f(5m) > K
        m = (lo + hi) // 2
        if m + f(m) > K:
            hi = m
        else:
            lo = m + 1
    reached = lo                 # m = 0..lo-1 có f(5m) <= K
    return (K + 1) - reached

for K in range(0, 2000):
    assert count_skipped_ans(K) == sum(1 for k in range(K + 1) if table[k] == 0), K
print("count_skipped(10^9) =", count_skipped_ans(10**9))

# Bài 5
def preimage_p_ans(k, p):
    def vp(x):
        c = 0
        while x:
            x //= p
            c += x
        return c
    lo, hi = 0, p * k
    while lo <= hi:
        mid = (lo + hi) // 2
        z = vp(mid)
        if z == k:
            return p
        if z < k:
            lo = mid + 1
        else:
            hi = mid - 1
    return 0

for p in (2, 3, 5, 7):
    tab = Counter(legendre(x, p) for x in range(4000))
    for k in range(0, 300):
        assert preimage_p_ans(k, p) == tab[k], (p, k)
print("Tất cả đáp án đúng")

count_skipped(10^9) = 199999997


Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
def zeros(x):
    c = 0
    while x: x //= 5; c += x
    return c

lo, hi = 0, 5 * k
while lo <= hi:
    mid = (lo + hi) // 2
    z = zeros(mid)
    if z == k: return 5
    if z < k: lo = mid + 1
    else: hi = mid - 1
return 0
```

**Checklist trước khi submit:**

- [ ] `zeros(x)` cộng **tất cả** `x//5 + x//25 + ...`.
- [ ] Cận trên đủ lớn: `hi = 5k` (với nhị phân trên `x`).
- [ ] `k = 0` → `5` (x = 0..4).
- [ ] Đáp án chỉ là `0` hoặc `5` — không trả số khác.
- [ ] Không tính `x!` thật.